# Experience classifier, version 2

Version 1 (4 epochs, unweighted loss) collapsed to the two largest classes (macro-F1 0.31, below the TF-IDF baseline of 0.54).
This version compares three set-ups on the **same** stratified 5-fold split and selects the one with the highest mean macro-F1 (rule fixed before running):

1. TF-IDF + logistic regression (baseline)
2. bert-base-uncased, 10 epochs, class-weighted loss, lr 3e-5
3. roberta-base, same settings

The selected model is then trained on all 582 reviews and applied to the corpus.

Runtime: T4 GPU, about 25 to 35 minutes. Upload `gold_standard_582.csv` and `corpus_for_inference.csv`. Send back `cv_metrics_v2.json`, `cv_predictions_v2.csv`, `corpus_predictions_v2.csv`.

In [ ]:
!pip -q install -U transformers accelerate scikit-learn
import transformers, torch, sklearn; print(transformers.__version__, torch.__version__, torch.cuda.is_available())

In [ ]:
from google.colab import files
up = files.upload()

In [ ]:
import numpy as np, pandas as pd, torch, random, json
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, cohen_kappa_score
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
SEED=42; EPOCHS=10; BS=16; LR=3e-5; MAXLEN=128
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
dev=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
gold=pd.read_csv('gold_standard_582.csv'); gold['text']=gold.text.astype(str)
y_all=gold.label.values
folds=list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(gold.text, y_all))

In [ ]:
def train_tf(name, texts, labels, seed):
    set_seed(seed)
    tok=AutoTokenizer.from_pretrained(name)
    model=AutoModelForSequenceClassification.from_pretrained(name, num_labels=6).to(dev)
    labels=np.asarray(labels); counts=np.bincount(labels, minlength=6).astype(float)
    w=torch.tensor(len(labels)/(6*np.maximum(counts,1)), dtype=torch.float).to(dev)
    lossf=torch.nn.CrossEntropyLoss(weight=w)
    enc=tok(list(texts), truncation=True, padding=True, max_length=MAXLEN, return_tensors='pt'); y=torch.tensor(labels)
    n=len(y); opt=torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps=EPOCHS*((n+BS-1)//BS); sch=get_linear_schedule_with_warmup(opt, int(0.1*steps), steps)
    model.train()
    for ep in range(EPOCHS):
        perm=torch.randperm(n)
        for i in range(0,n,BS):
            idx=perm[i:i+BS]
            logits=model(input_ids=enc['input_ids'][idx].to(dev), attention_mask=enc['attention_mask'][idx].to(dev)).logits
            loss=lossf(logits, y[idx].to(dev)); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step(); sch.step(); opt.zero_grad()
    return tok, model
@torch.no_grad()
def predict_tf(tok, model, texts, bs=128):
    model.eval(); P=[]; texts=list(texts)
    for i in range(0,len(texts),bs):
        e=tok(texts[i:i+bs], truncation=True, padding=True, max_length=MAXLEN, return_tensors='pt').to(dev)
        P.append(torch.softmax(model(**e).logits,-1).cpu().numpy())
    return np.vstack(P)
def run_cv(kind):
    oof=np.zeros(len(gold),dtype=int); rows=[]
    for k,(tr,te) in enumerate(folds):
        if kind=='tfidf_logreg':
            p=make_pipeline(TfidfVectorizer(ngram_range=(1,2),sublinear_tf=True), LogisticRegression(max_iter=2000,class_weight='balanced',C=5))
            p.fit(gold.text.iloc[tr], y_all[tr]); pr=p.predict(gold.text.iloc[te])
        else:
            tok,m=train_tf(kind, gold.text.iloc[tr], y_all[tr], SEED+k)
            pr=predict_tf(tok,m,gold.text.iloc[te]).argmax(1); del m; torch.cuda.empty_cache()
        oof[te]=pr
        rows.append({'fold':k+1,'accuracy':accuracy_score(y_all[te],pr),'macro_f1':f1_score(y_all[te],pr,average='macro')})
        print(kind, rows[-1])
    f=pd.DataFrame(rows)
    return {'folds':rows,'mean':f.mean(numeric_only=True).to_dict(),'std':f.std(numeric_only=True).to_dict(),
            'pooled_report':classification_report(y_all,oof,digits=3,output_dict=True),
            'kappa_vs_gold':cohen_kappa_score(y_all,oof),'confusion_matrix':confusion_matrix(y_all,oof,labels=range(6)).tolist()}, oof

In [ ]:
results={}; oofs={}
for kind in ['tfidf_logreg','bert-base-uncased','roberta-base']:
    results[kind], oofs[kind] = run_cv(kind)
    print(kind, 'mean macro-F1', round(results[kind]['mean']['macro_f1'],3), 'sd', round(results[kind]['std']['macro_f1'],3))
best=max(results, key=lambda k: results[k]['mean']['macro_f1']); print('SELECTED:', best)
print(classification_report(y_all, oofs[best], digits=3))

In [ ]:
corpus=pd.read_csv('corpus_for_inference.csv'); corpus['text']=corpus.text.astype(str)
if best=='tfidf_logreg':
    p=make_pipeline(TfidfVectorizer(ngram_range=(1,2),sublinear_tf=True), LogisticRegression(max_iter=2000,class_weight='balanced',C=5)).fit(gold.text, y_all)
    P=p.predict_proba(corpus.text)
else:
    tok,m=train_tf(best, gold.text, y_all, SEED); P=predict_tf(tok,m,corpus.text)
corpus['pred']=P.argmax(1); corpus['prob']=P.max(1)
corpus[['doc_id','dest','pred','prob']].to_csv('corpus_predictions_v2.csv', index=False)
print(corpus.pred.value_counts(normalize=True).sort_index())
json.dump({'selected':best,'results':results,'settings':{'epochs':EPOCHS,'batch':BS,'lr':LR,'maxlen':MAXLEN,'seed':SEED,'loss':'class-weighted cross-entropy','transformers':transformers.__version__,'torch':torch.__version__}},
          open('cv_metrics_v2.json','w'), indent=2)
pd.DataFrame({'ID':gold.ID,'label':y_all, **{f'pred_{k}':v for k,v in oofs.items()}}).to_csv('cv_predictions_v2.csv', index=False)

In [ ]:
files.download('cv_metrics_v2.json'); files.download('cv_predictions_v2.csv'); files.download('corpus_predictions_v2.csv')